# M13 — LoRanPAC equal-budget challenger (train-only)

Runs the released LoRanPAC continual-TSVD recurrence on the exact M6/M11 RanPAC development stream. Rank caps are derived from P2B/adaptive state bytes before accuracy is evaluated. Run every cell in order; `test.pt` is never materialized. Completed width/budget units are reusable if the long cell is restarted.

In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_COMMIT='002110041b448d64d09e5c80e7a7a004ab3136a8'
WORK_DIR='/content/SOHO-CL'
FEATURE_CACHE_DIR='/content/srq_m13_cifar_features'
OUTPUT_DIR='/content/srq_m13_loranpac_output'
EXPECTED_M6_NAME='srq_generalization_m6_width_sweep_train_only.zip'
EXPECTED_M6_SHA='b2739b9da023ebd2eedb6fdfe01c394e94f252773e847533b35350021c3d239e'
EXPECTED_M11_NAME='srq_generalization_m11_adaptive_precision_train_only.zip'
EXPECTED_M11_SHA='65ce03df4da7041833014628b59aac1167f77bde2d64348b9a8a1e4fe09370a7'
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Exact checkout, dependencies, GPU check, and canonical-LF source verification.
import hashlib,json,os,shutil,subprocess,sys
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> GPU.'
def sha_raw(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
EXPECTED={
 'configs/srq_generalization_m13_loranpac_train_only.json':'0fe77c32c6c242bdae70f3d720a54c624e25c0b1099ad3e263c03d8e178698dc',
 'tools/srq_generalization_m13.py':'a8b090ca4bf8a994e81222c63500fec79e3a1233065f416c886fd86cde24b82e',
 'methods/frontends/loranpac.py':'b468d98981671876bbd223e62ab34ad8430d93f315662b0dabd36ab28ba980f7',
 'methods/frontends/ranpac.py':'6b94532f607d245c0d148d09c1a36b44dbbf964f4ee9cbf05ba6f64826a90e60',
 'tools/srq_generalization_m6.py':'bad119dca8b2c6e78200c81917c8e8b03a5c50923135f951d8723fd5afd2ae61',
 'tools/srq_generalization_m5.py':'4d08e27a825fb59d300ee5909542bca4bd550a8558bb137159a176f353739a84',
 'tools/srq_generalization_m4.py':'84302805f6c71475cfcd3f7c9f148700198e96879cac6c795ef0f1bbc0f4c29e',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'models/backbone.py':'941e449dc6e66ca4018fb0d3ab3218d97ec97f498b557ed220c8332e75850a46',
 'utils/data_utils.py':'3cf85993e231b068ad5ae2f96be608b2e50e9c52f98fb2387fd3badfb44b6764',
 'utils/train_utils.py':'e24983bd3042ad82ec069916ba2853cf1c818cb2911ce193710c8ccd70e86bda'}
for path,expected in EXPECTED.items(): assert sha_source(path)==expected,(path,sha_source(path),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Repository must start clean.'
CONFIG='configs/srq_generalization_m13_loranpac_train_only.json'
RUNNER='tools/srq_generalization_m13.py'
print('GPU:',torch.cuda.get_device_name(0))
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('M13 SOURCE LOCK: PASS')

In [ ]:
# Mathematical, state-accounting, and protocol gates before data download.
command=[sys.executable,'-B','-m','pytest','-q','-p','no:cacheprovider','tests/test_loranpac_analytic_frontend.py','tests/test_srq_generalization_m13.py','tests/test_ranpac_analytic_frontend.py','tests/test_tail_fly_math.py']
completed=subprocess.run(command)
assert completed.returncode==0,'M13 local gates failed; return the complete traceback.'
print('M13 LOCAL GATES: PASS')

In [ ]:
# Upload the exact M6 and M11 source artifacts; do not unzip or rename them.
from google.colab import files
uploaded=files.upload()
for name,expected in [(EXPECTED_M6_NAME,EXPECTED_M6_SHA),(EXPECTED_M11_NAME,EXPECTED_M11_SHA)]:
    assert name in uploaded,f'Upload {name} exactly.'
    uploaded_path=Path(name).resolve(); destination=Path('/content')/name
    if destination.exists(): destination.unlink()
    shutil.move(str(uploaded_path),str(destination))
    assert sha_raw(destination)==expected,(name,sha_raw(destination),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Artifact upload contaminated the repository.'
SOURCE_M6_ARTIFACT=str(Path('/content')/EXPECTED_M6_NAME)
SOURCE_M11_ARTIFACT=str(Path('/content')/EXPECTED_M11_NAME)
print('M6/M11 ARTIFACT LOCKS: PASS')

In [ ]:
# Download the locked checkpoint and processed CIFAR-100 source.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha_raw(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('CHECKPOINT:',CHECKPOINT_PATH)
print('CIFAR ROOT:',CIFAR_ROOT)

In [ ]:
# Materialize TRAIN features only.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b','--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_m13','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START',flush=True); subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
print('TRAIN CACHE READY:',metadata.get('train_shape'),'| test.pt absent')

In [ ]:
# Long, resumable M13 run. Re-run this cell after an interruption; completed units are reused.
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--source-m6-artifact',SOURCE_M6_ARTIFACT,'--source-m11-artifact',SOURCE_M11_ARTIFACT,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
print('M13 START: LoRanPAC TSVD at P2B/adaptive byte budgets, widths 10k/20k.',flush=True)
completed=subprocess.run(command)
RUN_RETURN_CODE=completed.returncode
result_path=Path(OUTPUT_DIR)/'m13_results.json'
if not result_path.is_file():
    print('COMPLETED UNITS:',[p.name for p in Path(OUTPUT_DIR).glob('m13_unit_*.json')])
    raise RuntimeError('M13 stopped inside a unit. Re-run this cell; completed units will be reused.')
result=json.loads(result_path.read_text())
print('STATUS:',result['status'])
print('SUMMARY:',json.dumps(result['summary'],indent=2))
print('GATES:',json.dumps(result['gates'],indent=2))

In [ ]:
# Compact source-versus-LoRanPAC table. Both LoRanPAC heads were fixed before evaluation.
import pandas as pd
rows=[]
for width,comparison in result['source_comparisons'].items():
    for method,item in comparison.items(): rows.append({'width':int(width),'method':method,'budget':'source','rank':None,'head':'source','AIA':item['validation_aia_percent'],'A_final':item['final_validation_accuracy_percent'],'state_MiB':item['final_total_persistent_bytes']/2**20})
for unit in result['units']:
    for head in ('official_ridge0','matched_m6_ridge'): rows.append({'width':unit['width'],'method':'LoRanPAC-TSVD','budget':unit['budget_target'],'rank':unit['rank_contract']['derived_max_rank'],'head':head,'AIA':unit['validation_aia_percent'][head],'A_final':unit['final_validation_accuracy_percent'][head],'state_MiB':unit['final_total_persistent_bytes']/2**20})
display(pd.DataFrame(rows).sort_values(['width','state_MiB','method','head']))

In [ ]:
# Vector Pareto and rank-trajectory figures generated only from locked results.
import matplotlib.pyplot as plt
frame=pd.DataFrame(rows)
fig,axes=plt.subplots(1,2,figsize=(11,4.1))
for width in sorted(frame.width.unique()):
    part=frame[frame.width==width]
    axes[0].scatter(part.state_MiB,part.AIA,label=f'{width//1000}k')
    for _,row in part.iterrows(): axes[0].annotate(f"{row['method']}\n{row['head']}",(row.state_MiB,row.AIA),fontsize=6)
for unit in result['units']:
    axes[1].plot([r['task'] for r in unit['records']],[r['effective_rank'] for r in unit['records']],marker='o',label=f"{unit['width']//1000}k/{unit['budget_target']}")
axes[0].set_xlabel('Persistent state (MiB)'); axes[0].set_ylabel('Validation AIA (%)')
axes[1].set_xlabel('Task'); axes[1].set_ylabel('Effective TSVD rank')
for ax in axes: ax.grid(True,alpha=.25); ax.legend(fontsize=7)
fig.tight_layout(); plot_path=Path(OUTPUT_DIR)/'m13_loranpac_pareto.svg'
fig.savefig(plot_path,format='svg'); plt.show(); plt.close(fig)
assert plot_path.is_file()

In [ ]:
# Export evidence whether the scientific outcome favors SRQ or LoRanPAC.
bundle=Path('/content/srq_generalization_m13_loranpac_train_only')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
for source,name in [(Path(OUTPUT_DIR)/'m13_results.json','m13_results.json'),(Path(OUTPUT_DIR)/'m13_accuracy_state.csv','m13_accuracy_state.csv'),(Path(OUTPUT_DIR)/'m13_loranpac_pareto.svg','m13_loranpac_pareto.svg'),(Path(CONFIG),'config.json'),(Path('docs/research/SRQ_GENERALIZATION_M13_RUNBOOK.md'),'runbook.md')]: shutil.copy2(source,bundle/name)
for unit in Path(OUTPUT_DIR).glob('m13_unit_*.json'): shutil.copy2(unit,bundle/unit.name)
manifest={'schema_version':1,'status':result['status'],'uses_test_set':False,'git_commit':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),'source_m6_sha256':EXPECTED_M6_SHA,'source_m11_sha256':EXPECTED_M11_SHA,'files':{p.name:sha_raw(p) for p in bundle.iterdir()}}
(bundle/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
EXPORT_PATH='/content/srq_generalization_m13_loranpac_train_only.zip'
archive=shutil.make_archive(str(bundle),'zip',root_dir=bundle)
assert archive==EXPORT_PATH
print('ARTIFACT:',archive,'SHA-256:',sha_raw(archive))
files.download(archive)
assert RUN_RETURN_CODE==0 and result['status']=='PASS_M13_LORANPAC_TRAIN_ONLY','M13 integrity/numerical gates failed; keep the artifact and do not add an accuracy-based retry.'